In [1]:
import cv2
import numpy as np
import pickle
from collections import deque
from tensorflow.keras.models import load_model
from fsl_preprocessing import normalize_landmarks
  

# Load model
model = load_model("models/dynamic/subset_lstm_model.h5", compile=False)

# Load label encoder
with open("models/dynamic/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# Load preprocessing config
with open("models/dynamic/preprocess_config.pkl", "rb") as f:
    config = pickle.load(f)

SEQ_LENGTH = config["seq_length"]

sequence = deque(maxlen=SEQ_LENGTH)


In [2]:
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)


In [6]:
import cv2
import numpy as np
import pickle
from collections import deque
from tensorflow.keras.models import load_model
from fsl_preprocessing import normalize_landmarks
  

# Load model
model = load_model("models/dynamic/subset_lstm_model.h5", compile=False)

# Load label encoder
with open("models/dynamic/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# Load preprocessing config
with open("models/dynamic/preprocess_config.pkl", "rb") as f:
    config = pickle.load(f)

SEQ_LENGTH = config["seq_length"]

sequence = deque(maxlen=SEQ_LENGTH)


import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("Camera failed to open")
    exit()


from collections import deque, Counter

sequence = deque(maxlen=SEQ_LENGTH)
predictions = deque(maxlen=10)   # limit smoothing window

CONF_THRESHOLD = 0.5



print("Starting webcam...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    display_label = "..."

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]

        landmarks = []
        for lm in hand_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z])

        landmarks = normalize_landmarks(landmarks)
        sequence.append(landmarks)

        if len(sequence) == SEQ_LENGTH:
            input_data = np.expand_dims(sequence, axis=0)
            prediction = model.predict(input_data, verbose=0)
            print("Raw prediction:", prediction)

            confidence = np.max(prediction)
            class_id = np.argmax(prediction)

            label = le.inverse_transform([class_id])[0]
            display_label = f"{label} ({confidence:.2f})"


            if confidence > CONF_THRESHOLD:
                predictions.append(class_id)

                # smoothing
                final_class = Counter(predictions).most_common(1)[0][0]
                display_label = le.inverse_transform([final_class])[0]

            print("Confidence:", confidence)
            print("Prediction:", prediction)
            print("Sequence len:", len(sequence))
            print("Confidence:", confidence if len(sequence)==SEQ_LENGTH else "N/A")
    else:
        sequence.clear()
        predictions.clear()

    cv2.putText(frame,
                display_label,
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2)

    cv2.imshow("Dynamic Sign Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Starting webcam...


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Raw prediction: [[0.42742687 0.22889382 0.34367928]]
Confidence: 0.42742687
Prediction: [[0.42742687 0.22889382 0.34367928]]
Sequence len: 30
Confidence: 0.42742687


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Raw prediction: [[0.44396684 0.22744204 0.3285911 ]]
Confidence: 0.44396684
Prediction: [[0.44396684 0.22744204 0.3285911 ]]
Sequence len: 30
Confidence: 0.44396684
Raw prediction: [[0.46834835 0.22817054 0.30348116]]
Confidence: 0.46834835
Prediction: [[0.46834835 0.22817054 0.30348116]]
Sequence len: 30
Confidence: 0.46834835


...

Raw prediction: [[0.5303171  0.23480918 0.23487374]]
Confidence: 0.5303171
Prediction: [[0.5303171  0.23480918 0.23487374]]
Sequence len: 30
Confidence: 0.5303171


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Raw prediction: [[0.538459   0.23832944 0.22321156]]
Confidence: 0.538459
Prediction: [[0.538459   0.23832944 0.22321156]]
Sequence len: 30
Confidence: 0.538459
Raw prediction: [[0.5462792  0.23744684 0.21627398]]
Confidence: 0.5462792
Prediction: [[0.5462792  0.23744684 0.21627398]]
Sequence len: 30
Confidence: 0.5462792


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


KeyboardInterrupt: 

In [8]:
print("Sample sequence:")
print(sequence[:2])

Sample sequence:


TypeError: sequence index must be integer, not 'slice'